#### EDA 08: Geolocation table
**File:** `olist_geolocation_dataset.csv`  |  **Goal:** understand the zip prefix to coordinates lookup, and what has to be cleaned before it is used.

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import unicodedata

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data

geo_raw = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
geo = geo_raw.copy()                              # all changes happen on the copy, raw stays untouched

print("shape:", geo.shape)
print(geo.dtypes)
geo.head(3)

shape: (1000163, 5)
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


- 1,000,163 rows and 5 columns: zip prefix, latitude, longitude, city and state. It is the largest table, with about 10 times the rows of orders.
- The zip prefix is stored as a number.
- One row is one coordinate point, not one zip prefix. The table is a point cloud per prefix.

#### Missing values and duplicates
**Question:** are there gaps, and how many rows are repeats?

In [2]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(geo),
                        "n_missing": geo.isna().sum(),
                        "pct_missing": (geo.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

dups = geo.duplicated().sum()
print("fully duplicated rows            :", dups, f"({dups / len(geo) * 100:.1f}%)")
print("distinct rows                    :", len(geo.drop_duplicates()))
print("repeated (prefix, lat, lng)      :", geo.duplicated(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]).sum())
print("distinct (prefix, lat, lng)      :", len(geo.drop_duplicates(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])))

,n_rows,n_missing,pct_missing
geolocation_zip_code_prefix,1000163,0,0.0
geolocation_lat,1000163,0,0.0
geolocation_lng,1000163,0,0.0
geolocation_city,1000163,0,0.0
geolocation_state,1000163,0,0.0


fully duplicated rows            : 261831 (26.2%)


distinct rows                    : 738332
repeated (prefix, lat, lng)      : 280009


distinct (prefix, lat, lng)      : 720154


What the output says
- No column has a missing value.
- The table has 10,00,163 rows.
- 2,61,831 rows are exact copies of another row. That is 26.2%.
- After removing the copies 7,38,332 rows are left.

A second check
- Each row has a zip prefix, a latitude and a longitude. Together they point to one place.
- 2,80,009 rows point to a place that already appeared earlier.
- 2,61,831 of these are the exact copies from above.
- The other 18,178 are not exact copies. They have the same place, but the city or state is written differently (for example sao paulo and são paulo).
- Only 7,20,154 different places are left.

In [3]:
geo = geo.drop_duplicates().reset_index(drop=True)      # working copy without exact copies
print("rows after removing exact copies:", len(geo))

rows after removing exact copies: 738332


#### Coordinates
**Question:** are the coordinates inside Brazil?

In [6]:
display(geo[["geolocation_lat", "geolocation_lng"]].describe(percentiles=[0.01, 0.5, 0.99]).round(2))

# A box around Brazil (a rough check, not exact borders)
BR_LAT, BR_LNG = (-33.75, 5.27), (-73.99, -34.79)
in_brazil = geo["geolocation_lat"].between(*BR_LAT) & geo["geolocation_lng"].between(*BR_LNG)

outside = geo[~in_brazil]
print("rows outside the Brazil box    :", len(outside), f"({(~in_brazil).mean() * 100:.3f}%)")
print("distinct prefixes affected     :", outside["geolocation_zip_code_prefix"].nunique())
print("prefixes with NO point inside the box:",
      (geo.assign(ok=in_brazil).groupby("geolocation_zip_code_prefix")["ok"].sum() == 0).sum())
outside.head(5)

,geolocation_lat,geolocation_lng
count,738332.00,738332.00
mean,-21.00,-46.46
std,5.89,4.39
min,-36.61,-101.47
1%,-30.12,-57.69
50%,-22.87,-46.65
99%,-2.44,-34.93
max,45.07,121.11


rows outside the Brazil box    : 33 (0.004%)
distinct prefixes affected     : 21
prefixes with NO point inside the box: 5


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,zip5
272046,18243,28.008978,-15.536867,bom retiro da esperanca,SP,18243
355591,28165,41.614052,-8.411675,vila nova de campos,RJ,28165
355603,28155,-34.586422,-58.732101,santa maria,RJ,28155
355694,28155,42.439286,13.820214,santa maria,RJ,28155
356233,28333,38.381672,-6.328200,raposo,RJ,28333


- Latitude runs from -36.61 to 45.07 and longitude from -101.47 to 121.11. Brazil only covers about -33.75 to 5.27 and -73.99 to -34.79.
- Only 33 distinct rows (0.004%) fall outside the box, in 21 prefixes. The first five are listed above, with coordinates in Europe, the Atlantic and Argentina but a Brazilian state.
- 5 prefixes have no valid point at all, so they cannot be located.
- **Decision:** for the lookup, ignore points outside Brazil. They are wrong, and they are few (0.004% of rows).

#### City and state text
**Question:** are the city and state text consistent and clean?

In [8]:
def strip_accents(s):
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)).lower().strip()

city = geo["geolocation_city"]
print("rows with non-ASCII city text         :", (~city.map(str.isascii)).sum(), f"({(~city.map(str.isascii)).mean() * 100:.1f}%)")
print("distinct city names raw / no accents  :", city.nunique(), "/", city.map(strip_accents).nunique())
print("city names with symbols, digits or encoding:", city[city.str.contains(r"[\d@/\\()%]")].nunique())
print(city[city.str.contains(r"%")].drop_duplicates().tolist())

# How is Sao Paulo written?
print("\nwriting of Sao Paulo:")
display(city[city.map(strip_accents) == "sao paulo"].value_counts().rename("n_rows").to_frame())

by_prefix = geo.groupby("geolocation_zip_code_prefix").agg(n_cities=("geolocation_city", "nunique"), n_states=("geolocation_state", "nunique"))
print("prefixes with more than one city name:", (by_prefix["n_cities"] > 1).sum(), f"({(by_prefix['n_cities'] > 1).mean() * 100:.1f}%)")
print("prefixes with more than one state    :", (by_prefix["n_states"] > 1).sum())
print("state codes outside the 27 valid ones:", sorted(set(geo["geolocation_state"]) - set(
    "AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split())))

state_share = (geo_raw["geolocation_state"].value_counts(normalize=True) * 100).round(1)
print("\nshare of original rows, top 5 states:", state_share.head(5).to_dict())
conflict = by_prefix.index[by_prefix["n_states"] > 1]
geo[geo["geolocation_zip_code_prefix"].isin(conflict)].groupby("geolocation_zip_code_prefix")["geolocation_state"].unique().head(5).to_frame("states")

rows with non-ASCII city text         : 63436 (8.6%)


distinct city names raw / no accents  : 8011 / 5967
city names with symbols, digits or encoding: 19
['são joão do pau d%26apos%3balho', 'lambari d%26apos%3boeste']

writing of Sao Paulo:


,n_rows
geolocation_city,
sao paulo,79927
são paulo,19718


prefixes with more than one city name: 8556 (45.0%)
prefixes with more than one state    : 8
state codes outside the 27 valid ones: []

share of original rows, top 5 states: {'SP': 40.4, 'MG': 12.6, 'RJ': 12.1, 'RS': 6.2, 'PR': 5.8}


,states
geolocation_zip_code_prefix,
2116,"[SP, RN]"
4011,"[SP, AC]"
21550,"[RJ, AC]"
23056,"[RJ, AC]"
72915,"[GO, DF]"


- City text is messy. 63,436 rows (8.6%, after removing copies) have accents, and the same city appears in several spellings: São Paulo is `sao paulo` in 79,927 rows and `são paulo` in 19,718 rows.
- Removing accents cuts the distinct city names from 8,011 to 5,967.
- 19 city names contain symbols, digits or brackets, and 2 contain web-encoded characters (for example `são joão do pau d%26apos%3balho`), which is a broken text encoding.
- 8,556 prefixes (45.0%) have more than one city name. This comes from spelling variants, so the city text is not usable as a key.
- 8 prefixes appear under two states (for example 4011 as SP and AC, 21550 as RJ and AC). A prefix has one state, so some rows carry a wrong state.
- State codes are all valid. The 27 states are present, and SP has 40.4% of the original rows, then MG 12.6%, RJ 12.1%, RS 6.2% and PR 5.8%.
- **Decision:** ignore the city text. Use only the prefix and, for the state, the most frequent state per prefix.